In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense ,Dropout, GRU
import os

In [3]:
with open(r"C:\Users\Mayuri\Desktop\ML project\datasets\The Bhopal gas disaster.txt") as f:
    corpus = f.read().lower()

In [26]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])
total_words = len(tokenizer.word_index) + 1

In [5]:
# Create input sequences using sliding window
input_sequences = []
words = corpus.split()
# for i in range(1, len(words)):
#     n_gram_sequence = words[:i+1]
#     token_list = tokenizer.texts_to_sequences([' '.join(n_gram_sequence)])[0]
#     input_sequences.append(token_list)

sequence_length = 5  # or 10
for i in range(sequence_length, len(words)):
    seq = words[i - sequence_length:i + 1]
    token_list = tokenizer.texts_to_sequences([' '.join(seq)])[0]
    input_sequences.append(token_list)

In [6]:
# Pad sequences and split into input and label
max_seq_len = max([len(x) for x in input_sequences])
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))
print(input_sequences)

[[   0    0    0 ...   39   17   14]
 [   0    0    0 ...   17   14  229]
 [   0    0    0 ...   14  229    7]
 ...
 [   0    0    0 ... 1964   19 1965]
 [   0    0    0 ...   19 1965    3]
 [   0    0    0 ...    3  720 1966]]


In [18]:
max_seq_len 

12

In [7]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]
# y = tf.keras.utils.to_categorical(y, num_classes=total_words)


In [8]:
# Define model
model = Sequential()
model.add(Embedding(input_dim=total_words, output_dim=64))
# model.add(LSTM(64))
model.add(GRU(64))
# model.add(Dropout(0.2))
# model.add(LSTM(32))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [9]:
print("Total input sequences:", len(input_sequences))
print("X shape:", X.shape, "y shape:", y.shape)

Total input sequences: 6563
X shape: (6563, 11) y shape: (6563,)


In [10]:
# Train model
model.fit(X, y, epochs=150, batch_size=64, validation_split=0.2)


Epoch 1/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.0600 - loss: 7.3092 - val_accuracy: 0.0990 - val_loss: 6.5816
Epoch 2/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.0699 - loss: 6.3234 - val_accuracy: 0.0990 - val_loss: 6.6877
Epoch 3/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.0746 - loss: 6.2253 - val_accuracy: 0.0990 - val_loss: 6.7062
Epoch 4/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.0722 - loss: 6.1508 - val_accuracy: 0.0982 - val_loss: 6.8338
Epoch 5/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.0765 - loss: 6.0368 - val_accuracy: 0.0990 - val_loss: 6.8867
Epoch 6/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.0721 - loss: 5.9830 - val_accuracy: 0.0944 - val_loss: 6.9342
Epoch 7/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.0954 - loss: 5.7764 - val_accuracy: 0.1028 - val_loss: 6.9465
Epoch 8/150
83/83 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.1020 - loss: 5.6546 - val_accuracy: 0.

In [11]:
loss, acc = model.evaluate(X, y)
print(f"Final Training Accuracy: {acc*100:.2f}%")

206/206 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9661 - loss: 0.3842
Final Training Accuracy: 80.57%


In [12]:
 import pickle
 with open('model_GRU.pkl', 'wb') as f:
     pickle.dump(model, f)

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

def predict_next_word(seed_text, tokenizer, model, max_sequence_len):
    # Step 1: Convert seed_text to token list
    token_list = tokenizer.texts_to_sequences([seed_text])[0]

    # Step 2: Pad the token list to max_sequence_len - 1
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

    # Step 3: Predict the probabilities for the next word
    predicted_probs = model.predict(token_list, verbose=0)[0]

    # Step 4: Get the index of the most probable word
    predicted_index = np.argmax(predicted_probs)

    # Step 5: Convert index back to word
    for word, index in tokenizer.word_index.items():
        if index == predicted_index:
            return word
    return ""

In [14]:
max_sequence_len=X.shape[1]+1

In [15]:

seed_text = "the tragrdy of the"
next_word = predict_next_word(seed_text, tokenizer, model, max_sequence_len)
print(f"{seed_text} {next_word}")

the tragrdy of the government


In [16]:
seed_text = "disaster is caused due to the"
next_word = predict_next_word(seed_text, tokenizer, model, max_sequence_len)
print(f"{seed_text} {next_word}")

disaster is caused due to the gases


In [17]:
import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
